In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/datasets/train_transaction.csv')

In [10]:
!pip install optuna mlflow imbalanced-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report

from imblearn.over_sampling import SMOTE

import tensorflow as tf

import optuna
import mlflow

In [12]:
missing_percent = df.isnull().mean()*100

missing_percent.sort_values(ascending=False).head(20)

,0
D7,97.60
D13,96.22
dist2,96.04
D12,95.34
D14,94.82
D6,94.50
D9,89.70
D8,89.70
V151,87.22
V140,87.22


In [13]:
missing_ratio = df.isnull().mean()

drop_cols = missing_ratio[missing_ratio > 0.8].index

df = df.drop(columns=drop_cols)

print(df.shape)

(5000, 226)


In [14]:
y = df['isFraud']

X = df.drop('isFraud', axis=1)

In [15]:
cat_cols = X.select_dtypes(include='object').columns

print(cat_cols)

Index(['ProductCD', 'card4', 'card6', 'P_emaildomain', 'M1', 'M2', 'M3', 'M4',
       'M5', 'M6', 'M7', 'M8', 'M9'],
      dtype='object')


In [17]:
for col in cat_cols:

    X[col] = X[col].fillna("missing")

    le = LabelEncoder()

    X[col] = le.fit_transform(
        X[col].astype(str)
    )

In [18]:
num_cols = X.select_dtypes(include=np.number).columns

for col in num_cols:
    X[col] = X[col].fillna(
        X[col].median()
    )

In [19]:
scaler = StandardScaler()

X = scaler.fit_transform(X)

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [21]:
smote = SMOTE(
    random_state=42
)

X_train, y_train = smote.fit_resample(
    X_train,
    y_train
)

print(y_train.value_counts())

isFraud
0    3913
1    3913
Name: count, dtype: int64


In [22]:
model = tf.keras.Sequential([

    tf.keras.layers.Dense(
        256,
        activation='relu'
    ),

    tf.keras.layers.Dropout(
        0.3
    ),

    tf.keras.layers.Dense(
        128,
        activation='relu'
    ),

    tf.keras.layers.Dropout(
        0.3
    ),

    tf.keras.layers.Dense(
        64,
        activation='relu'
    ),

    tf.keras.layers.Dense(
        1,
        activation='sigmoid'
    )

])

In [23]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.AUC(name='auc')
    ]
)

In [24]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=1024
)

Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 116ms/step - auc: 0.7097 - loss: 0.6035 - val_auc: 0.0000e+00 - val_loss: 0.6191
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - auc: 0.8867 - loss: 0.4278 - val_auc: 0.0000e+00 - val_loss: 0.4514
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - auc: 0.9234 - loss: 0.3518 - val_auc: 0.0000e+00 - val_loss: 0.3494
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - auc: 0.9484 - loss: 0.2884 - val_auc: 0.0000e+00 - val_loss: 0.3083
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - auc: 0.9636 - loss: 0.2423 - val_auc: 0.0000e+00 - val_loss: 0.2041
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - auc: 0.9730 - loss: 0.2109 - val_auc: 0.0000e+00 - val_loss: 0.1533
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - auc: 0.9783 - loss: 0.1857 - val_auc: 0.0000e+00 - val_loss: 0.1259
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - auc: 0.9855 - loss: 0.1539 - val_auc: 0.0000e+00 - val_loss: 0.1147
Epoch 9/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/ste

In [25]:
pred_prob = model.predict(X_test)

auc = roc_auc_score(
    y_test,
    pred_prob
)

print("ROC AUC =", auc)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
ROC AUC = 0.6583007994050939


In [26]:
pred = (pred_prob > 0.5).astype(int)

print(
    classification_report(
        y_test,
        pred
    )
)

              precision    recall  f1-score   support

           0       0.98      0.97      0.98       978
           1       0.12      0.18      0.14        22

    accuracy                           0.95      1000
   macro avg       0.55      0.58      0.56      1000
weighted avg       0.96      0.95      0.96      1000



Optuna hyperparameter

In [27]:
def objective(trial):

    units1 = trial.suggest_int(
        "units1",
        64,
        512
    )

    units2 = trial.suggest_int(
        "units2",
        32,
        256
    )

    dropout = trial.suggest_float(
        "dropout",
        0.1,
        0.5
    )

    lr = trial.suggest_float(
        "lr",
        1e-5,
        1e-2,
        log=True
    )

    model = tf.keras.Sequential([

        tf.keras.layers.Dense(
            units1,
            activation='relu'
        ),

        tf.keras.layers.Dropout(
            dropout
        ),

        tf.keras.layers.Dense(
            units2,
            activation='relu'
        ),

        tf.keras.layers.Dropout(
            dropout
        ),

        tf.keras.layers.Dense(
            1,
            activation='sigmoid'
        )
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=lr
        ),
        loss='binary_crossentropy',
        metrics=['AUC']
    )

    model.fit(
        X_train,
        y_train,
        epochs=5,
        batch_size=2048,
        verbose=0
    )

    pred = model.predict(
        X_test,
        verbose=0
    )

    auc = roc_auc_score(
        y_test,
        pred
    )

    return auc

In [28]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=20
)

print(study.best_params)
print(study.best_value)

[I 2026-06-18 23:45:30,106] A new study created in memory with name: no-name-6c6a6d76-aa0b-467c-aa2f-b91a9b7aa571
[I 2026-06-18 23:45:32,460] Trial 0 finished with value: 0.6613682840676706 and parameters: {'units1': 70, 'units2': 36, 'dropout': 0.3728810884383009, 'lr': 0.002375557879298955}. Best is trial 0 with value: 0.6613682840676706.
[I 2026-06-18 23:45:35,877] Trial 1 finished with value: 0.6068042387060792 and parameters: {'units1': 443, 'units2': 241, 'dropout': 0.16197433785944135, 'lr': 0.009227349266936788}. Best is trial 0 with value: 0.6613682840676706.
[I 2026-06-18 23:45:39,389] Trial 2 finished with value: 0.6281372002230897 and parameters: {'units1': 376, 'units2': 33, 'dropout': 0.3643080949283709, 'lr': 0.007298594088768507}. Best is trial 0 with value: 0.6613682840676706.
[I 2026-06-18 23:45:42,682] Trial 3 finished with value: 0.4704870793827849 and parameters: {'units1': 140, 'units2': 83, 'dropout': 0.28371320268936817, 'lr': 3.480072507875124e-05}. Best is tri

{'units1': 290, 'units2': 169, 'dropout': 0.27473372176577204, 'lr': 0.00010755082660254729}
0.6953894775980666


In [29]:
with mlflow.start_run():

    mlflow.log_params(
        study.best_params
    )

    mlflow.log_metric(
        "roc_auc",
        study.best_value
    )

    mlflow.tensorflow.log_model(
        model,
        "fraud_model"
    )

2026/06/18 23:48:33 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/18 23:48:33 INFO mlflow.store.db.utils: Updating database tables
2026/06/18 23:48:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/18 23:48:36 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
